# 02 – Paarungsgeometrie nach DIN ISO 21771

**Ziel.** Dieses Notebook dokumentiert die Größen eines Außenradpaares: Betriebseingriffswinkel, Achsabstand und Summe der Profilverschiebungsfaktoren, Wälzkreise, Teilungen, aktive Bereiche der Zahnflanken, Eingriffsstrecke, Überdeckungen und Gleitverhältnisse. Es sichert das Modul `gearcore.pair` und den Paarvergleich in `gearcore.parity` ab.

**Regeln.** Das Notebook ist ausführbare Dokumentation (ADR-104): Es ruft `gearcore` auf und definiert selbst keine Formeln. Jede Aussage trägt Norm, Ausgabe, Abschnitt, Gleichung und Seite. Normtext wird nicht wörtlich wiedergegeben; die Formeln sind eigene Transkriptionen in der Schreibweise der aktuell gültigen Norm. Programmnamen, Formelzeichen und Benennungen stehen in der Größen-Registry (`data/quantities.yaml`). Gerechnet wird durchgängig mit doppelter Genauigkeit.

**Quellen.**

| Schlüssel in `sources.yaml` | Norm | Inhalt in diesem Notebook |
|---|---|---|
| `ISO21771:2014` | DIN ISO 21771:2014-08 | §4.4 Teilungen, §4.5 Durchmesser, §5.2 Paarungsgrößen, §5.3 Summe der Profilverschiebungsfaktoren, §5.4 Zahneingriff, §5.6 Gleitverhältnisse, Anhang NB (Korrekturen) |
| `ISOTR6336-30:2022` | ISO/TR 6336-30:2022 | Rechenbeispiel 1 (Anhang A) als Referenz |
| `DIN3960:1987` | DIN 3960:1987-03 (zurückgezogen) | Vergleich: Rechengrundlage von STplus |

In [ ]:
import inspect

import matplotlib.pyplot as plt
import pandas as pd
from pydantic import ValidationError

import gearcore
from gearcore import involute as iv
from gearcore import pair as pr
from gearcore.data import (
    load_expected_differences,
    load_norm_deviations,
    load_worked_example,
    printed_number,
    printed_tolerance,
    stplus_case_dirs,
)
from gearcore.errors import InputRangeError
from gearcore.models.common import Pair
from gearcore.models.inputs import GearInput, PairInput, SpanMeasurement, ToolProfile
from gearcore.parity import (
    ParityVerdict,
    compare_pair_geometry,
    compute_pair_from_data,
    detection_limits,
    pair_data,
    rows_repeating_an_input,
)
from gearcore.quantities import latex
from gearcore.trace import equations_of

pd.set_option("display.precision", 6)
print("gearcore", gearcore.__version__)

## 1 Welche Größen gegeben sind

Von Achsabstand $a_{\mathrm{w}}$ und den Nenn-Profilverschiebungsfaktoren $x_{\mathrm{1}}$, $x_{\mathrm{2}}$ sind **genau zwei** gegeben, der dritte Wert folgt (ADR-107):

| gegeben | es folgt | Gleichungen |
|---|---|---|
| $a_{\mathrm{w}}$, $x_{\mathrm{1}}$ | $x_{\mathrm{2}}$ | (54), (62) |
| $a_{\mathrm{w}}$, $x_{\mathrm{2}}$ | $x_{\mathrm{1}}$ | (54), (62) |
| $x_{\mathrm{1}}$, $x_{\mathrm{2}}$ | $a_{\mathrm{w}}$ | (55), (54) |
| $a_{\mathrm{w}}$, $x_{\mathrm{1}}$, $x_{\mathrm{2}}$ | Eingabefehler | – |

Ein gegebener Achsabstand ist fest: Er ist der Abstand der Bohrungen im Gehäuse. Eine überbestimmte Eingabe wird nicht stillschweigend aufgelöst. STplus behält in diesem Fall $x_{\mathrm{1}}$ und leitet $x_{\mathrm{2}}$ aus dem Achsabstand ab, ohne es zu melden; der `.ste`-Import macht diese Übersetzung ausdrücklich und nennt den verworfenen Wert. Sind $a_{\mathrm{w}}$ und nur $x_{\mathrm{2}}$ gegeben, leitet STplus $x_{\mathrm{1}}$ ab wie `gearcore` (Referenzfall `a_x2_only_z18_45`).

Eine **Zahnweite** wird in diesem Inkrement noch nicht ausgewertet. Zahnweite und Profilverschiebungsfaktor bestimmen einander (DIN 21773:2014-08 §7.2 Gl. (14), S. 13); dabei entspricht eine Zahnweite mit Zahndickenabmaß dem Erzeugungs-Profilverschiebungsfaktor $x_{\mathrm{E}}$ (DIN ISO 21771:2014-08 §7.4 Gl. (123), (124), S. 67), eine Nenn-Zahnweite dem Nennwert $x$. Wie eine eingegebene Zahnweite zu behandeln ist, wird mit den Prüfmaßen und Abmaßen geklärt (Inkremente 4 und 5, ADR-107). Bis dahin führt `gearcore` die Zahnweite mit und meldet, dass sie nicht verwendet wurde; nennt eine Eingabe nur Zahnweiten, wird sie gelesen, die Paarungsgeometrie aber nicht berechnet.

Das Rechenbeispiel 1 der ISO/TR 6336-30:2022 (Anhang A, Tabelle A.1, S. 43) gibt $a$ = 500 mm und $x_{\mathrm{2}}$ = 0 an und druckt $x_{\mathrm{1}}$ in Klammern. Abschnitt 4.2.18 (S. 9) derselben Norm erklärt die Klammer: Der Wert ist berechnet und steht nur zur Information da. Das Beispiel ist also der zweite Fall der Tabelle. Tabelle A.1 nennt außerdem die Zahnweiten beider Räder. Aus ihnen berechnet die Norm den Erzeugungs-Profilverschiebungsfaktor $x_{\mathrm{E1}}$ = 0,117 79 (S. 45); der Nennwert ist $x_{\mathrm{1}}$ = 0,145 22.

In [ ]:
example = load_worked_example("iso_tr_6336_30_2022_annex_a_example_1")
inputs, expected = example["inputs"], example["expected"]
shift = inputs["profile_shift_coefficient"]
assert shift["derived"] == ["pinion"], "x_1 steht in Klammern und wird nicht eingegeben"

# Tabelle A.1: h_fP = 1,4 m_n, rho_fP = 0,39 m_n, q = 0, s_pr = 0. Der Protuberanzbetrag folgt aus
# s_pr = pr - q (ISO 6336-3:2019, Tabelle 2, S. 4). Nichts davon ist vorbelegt.
tools = {
    role: ToolProfile(
        addendum_factor=inputs["basic_rack_dedendum_mm"]["factor_of_normal_module"][role],
        tip_radius_factor=inputs["basic_rack_fillet_radius_mm"]["factor_of_normal_module"][role],
        protuberance_mm=inputs["residual_fillet_undercut_mm"][role]
        + inputs["machining_allowance_mm"][role],
        machining_allowance_mm=inputs["machining_allowance_mm"][role],
    )
    for role in ("pinion", "wheel")
}
gears = {
    role: GearInput(
        number_of_teeth=inputs["number_of_teeth"][role],
        profile_shift_coefficient=None if role in shift["derived"] else shift[role],
        span=SpanMeasurement(
            span_measurement_mm=inputs["span_measurement_mm"][role],
            number_of_teeth_spanned=inputs["number_of_teeth_spanned"][role],
        ),
        face_width_mm=inputs["face_width_mm"][role],
        tip_diameter_mm=inputs["tip_diameter_mm"][role],
        tip_chamfer_radial_mm=0.0,  # Tabelle A.1 nennt keinen Kopfkantenbruch
        tool=tools[role],
    )
    for role in ("pinion", "wheel")
}
assert inputs["hand_of_helix"]["pinion"] == "left"  # positives beta = Ritzel rechtssteigend
pair = PairInput(
    normal_module_mm=inputs["normal_module_mm"]["value"],
    normal_pressure_angle_deg=inputs["normal_pressure_angle_deg"]["value"],
    helix_angle_deg=-inputs["helix_angle_deg"]["value"],
    centre_distance_mm=inputs["centre_distance_mm"]["value"],
    gears=Pair(pinion=gears["pinion"], wheel=gears["wheel"]),
)
result = pr.compute_pair_geometry(pair)

printed, decimals = printed_number(shift["printed"]["pinion"])
deviation = result.profile_shift_coefficient.pinion - printed
assert abs(deviation) <= printed_tolerance(decimals), "x_1 der Norm wird reproduziert"
generating = expected["generating_profile_shift_coefficient"]
assert generating["pinion"] < printed, "die Zahnweite des Beispiels führt auf x_E1 (mit Abmaß)"
pd.DataFrame(
    [
        {"Größe": "a_w in mm", "Wert": result.centre_distance_mm, "Herkunft": "gegeben"},
        {"Größe": "x_2", "Wert": result.profile_shift_coefficient.wheel, "Herkunft": "gegeben"},
        {
            "Größe": "x_1",
            "Wert": result.profile_shift_coefficient.pinion,
            "Herkunft": "folgt; Norm druckt " + shift["printed"]["pinion"],
        },
    ]
)

In [ ]:
# Alle drei Werte zusammen sind ein Eingabefehler, in der Eingabe und in der Rechnung.
both = Pair(
    pinion=gears["pinion"].model_copy(update={"profile_shift_coefficient": 0.14522}),
    wheel=gears["wheel"],
)
try:
    PairInput(**{**pair.model_dump(), "gears": both.model_dump()})
except ValidationError as error:
    message = error.errors()[0]["msg"]
assert "only two of the three may be given" in message
print(message)

## 2 Betriebseingriffswinkel, Achsabstand, Wälzkreise

**Normbezug.** DIN ISO 21771:2014-08: §5.2.1 Gl. (52), S. 38; §5.2.4 Gl. (54), (55), S. 39; §5.2.5 Gl. (56), (57), S. 39 mit der Korrektur im nationalen Anhang NB, S. 6; §5.3 Gl. (62), S. 41.

$$u = \frac{z_{\mathrm{2}}}{z_{\mathrm{1}}} \quad (52) \qquad \alpha_{\mathrm{wt}} = \arccos\!\left(\frac{(z_{\mathrm{1}} + z_{\mathrm{2}})\, m_{\mathrm{n}} \cos\alpha_{\mathrm{t}}}{2\, a_{\mathrm{w}} \cos\beta}\right) \quad (54)$$

$$\operatorname{inv}\alpha_{\mathrm{wt}} = \operatorname{inv}\alpha_{\mathrm{t}} + \frac{2 \tan\alpha_{\mathrm{n}}}{z_{\mathrm{1}} + z_{\mathrm{2}}}\,(x_{\mathrm{1}} + x_{\mathrm{2}}) \quad (55) \qquad \Sigma x = \frac{(z_{\mathrm{1}} + z_{\mathrm{2}})(\operatorname{inv}\alpha_{\mathrm{wt}} - \operatorname{inv}\alpha_{\mathrm{t}})}{2 \tan\alpha_{\mathrm{n}}} \quad (62)$$

$$d_{\mathrm{w1}} = \frac{d_{\mathrm{b1}}}{\cos\alpha_{\mathrm{wt}}} \qquad d_{\mathrm{w2}} = \frac{d_{\mathrm{b2}}}{\cos\alpha_{\mathrm{wt}}} \quad (56),\,(57)$$

Gl. (55) und (62) gelten für die spielfreie Paarung. Ist $a_{\mathrm{w}}$ gegeben, folgt $\alpha_{\mathrm{wt}}$ aus Gl. (54); sind beide Profilverschiebungsfaktoren gegeben, folgt $\alpha_{\mathrm{wt}}$ aus Gl. (55) mit der Umkehrung der Evolventenfunktion (Notebook 01) und $a_{\mathrm{w}}$ aus Gl. (54).

**Befund.** Der nationale Anhang NB korrigiert die erste Form von Gl. (56) und (57); `gearcore` verwendet die letzte Form $d_{\mathrm{w}} = d_{\mathrm{b}} / \cos\alpha_{\mathrm{wt}}$, die in beiden Fassungen gleich ist. Die Nummer (61) kommt in der Norm nicht vor: Auf Gl. (60) folgt Gl. (62), und unter (60) stehen zwei Formeln (Kopfspiel von Ritzel und Rad).

In [ ]:
PAIR_KEYS = [
    "gear_ratio",
    "transverse_working_pressure_angle_deg",
    "working_pitch_diameter_mm",
    "normal_pitch_mm",
    "transverse_pitch_mm",
    "transverse_base_pitch_mm",
    "transverse_contact_pitch_mm",
    "sap_diameter_mm",
    "length_of_path_of_contact_mm",
    "transverse_contact_ratio",
    "overlap_ratio",
    "total_contact_ratio",
]
norm_rows = []
for key in PAIR_KEYS:
    entry = expected[key]
    tolerance = printed_tolerance(entry["decimals"])
    value = getattr(result, key)
    for role in ("pinion", "wheel") if "pinion" in entry else (None,):
        ours = getattr(value, role) if role else value
        reference = entry[role] if role else entry["value"]
        assert abs(ours - reference) <= tolerance + 1e-9, (key, role, ours, reference)
        norm_rows.append(
            {
                "key": key,
                "Benennung der Norm": entry["description"],
                "Zeichen": entry["symbol"],
                "Rad": role or "–",
                "gearcore": ours,
                "Norm (gedruckt)": entry["printed"][role] if role else entry["printed"],
                "Seite": entry["page"],
            }
        )
norm_table = pd.DataFrame(norm_rows).set_index("key")
norm_table.loc[["gear_ratio", "transverse_working_pressure_angle_deg", "working_pitch_diameter_mm"]]

## 3 Teilungen

**Normbezug.** DIN ISO 21771:2014-08: §4.4.2.1 Gl. (23), S. 31; §4.4.2.2 Gl. (24), S. 32; §4.4.5 Gl. (28), S. 33; §4.4.5.1 Gl. (30), S. 33.

$$p_{\mathrm{t}} = \frac{\pi\, m_{\mathrm{n}}}{\cos\beta} \quad (23) \qquad p_{\mathrm{n}} = \pi\, m_{\mathrm{n}} \quad (24) \qquad p_{\mathrm{bt}} = p_{\mathrm{t}} \cos\alpha_{\mathrm{t}} \quad (28) \qquad p_{\mathrm{et}} = p_{\mathrm{bt}} \quad (30)$$

In [ ]:
norm_table.loc[
    [
        "normal_pitch_mm",
        "transverse_pitch_mm",
        "transverse_base_pitch_mm",
        "transverse_contact_pitch_mm",
    ]
]

## 4 Zahneingriff: aktive Flanke, Eingriffsstrecke, Überdeckungen

**Normbezug.** DIN ISO 21771:2014-08: §7.6 Gl. (127), S. 68; §5.4.1 Gl. (64) bis (69), S. 43; §5.4.5.2 Gl. (77), S. 45, Gl. (79), (80), S. 46; §5.4.5.3 Gl. (87), S. 46; §5.4.7.1 Gl. (90), S. 47; §5.4.7.3 Gl. (93), S. 48; §5.4.7.5 Gl. (97), S. 49.

$$d_{\mathrm{Fa}} = d_{\mathrm{a}} - 2\, h_{\mathrm{K}} \quad (127) \qquad d_{\mathrm{Nf1}} = \sqrt{\left(2\, a_{\mathrm{w}} \sin\alpha_{\mathrm{wt}} - \sqrt{d_{\mathrm{Fa2}}^2 - d_{\mathrm{b2}}^2}\right)^2 + d_{\mathrm{b1}}^2} \quad (64)$$

$$g_\alpha = \tfrac{1}{2}\left(\sqrt{d_{\mathrm{Na1}}^2 - d_{\mathrm{b1}}^2} + \sqrt{d_{\mathrm{Na2}}^2 - d_{\mathrm{b2}}^2} - 2\, a_{\mathrm{w}} \sin\alpha_{\mathrm{wt}}\right) \quad (77) \qquad g_{\mathrm{a}} = \tfrac{1}{2}\left(\sqrt{d_{\mathrm{Na}}^2 - d_{\mathrm{b}}^2} - d_{\mathrm{b}} \tan\alpha_{\mathrm{wt}}\right) \quad (80),\,(79)$$

$$\varepsilon_\alpha = \frac{g_\alpha}{p_{\mathrm{et}}} \quad (90) \qquad \varepsilon_\beta = \frac{b_{\mathrm{w}} \sin|\beta|}{\pi\, m_{\mathrm{n}}} \quad (93) \qquad \varepsilon_\gamma = \varepsilon_\alpha + \varepsilon_\beta \quad (97)$$

Der Beginn der aktiven Flanke $d_{\mathrm{Nf}}$ wird vom Kopf-Formkreis des Gegenrades bestimmt (Gl. (64), (65)). Ist der eigene Fuß-Formkreis $d_{\mathrm{Ff}}$ größer, gilt $d_{\mathrm{Nf}} = d_{\mathrm{Ff}}$ (Gl. (66), (67)), und der Kopf des Gegenrades wird nur bis $d_{\mathrm{Na}}$ genutzt (Gl. (68), (69)). Reicht der Kopf-Formkreis des Gegenrades über den Berührpunkt der Eingriffsgeraden am Grundkreis hinaus und ist kein Fuß-Formkreis gegeben, bricht die Rechnung ab (Eingriffsstörung).

**Was die Paarungsgeometrie nicht selbst bestimmt.**
- Die **Kopfkreisdurchmesser** sind Eingabe. Der Nennwert nach Gl. (33) braucht die Kopfhöhe des Bezugsprofils, die die werkzeugbasierte Eingabe nicht enthält; er steht über `with_nominal_tip_diameters` als ausdrückliche Wahl zur Verfügung.
- Die **Fuß-Formkreisdurchmesser** entstehen bei der Erzeugung (§7.6, Inkrement 3). Ohne sie meldet das Ergebnis, dass die Begrenzung nach Gl. (66) bis (69) nicht angewandt wurde.
- Bricht das Werkzeug die Kopfkante, entsteht auch der **Kopf-Formkreisdurchmesser** bei der Erzeugung; ohne ihn gilt Gl. (127) mit dem eingegebenen Radialbetrag $h_{\mathrm{K}}$.

**Befund.** Gl. (93) schreibt die Zahnbreite $b$, der Sprung-Überdeckungswinkel in Gl. (91) ist mit der genutzten Zahnbreite $b_{\mathrm{w}}$ definiert. `gearcore` rechnet mit $b_{\mathrm{w}}$ und setzt ohne Angabe eines Mittenversatzes $b_{\mathrm{w}} = \min(b_{\mathrm{1}}, b_{\mathrm{2}})$.

In [ ]:
print(list(dict.fromkeys(warning.code for warning in result.warnings)))
norm_table.loc[
    [
        "sap_diameter_mm",
        "length_of_path_of_contact_mm",
        "transverse_contact_ratio",
        "overlap_ratio",
        "total_contact_ratio",
    ]
]

In [ ]:
# Mit den Fuß-Formkreisdurchmessern, die die Norm druckt (S. 45), entfällt der Hinweis auf die
# fehlende Prüfung; sie liegen unter dem Beginn der aktiven Flanke, das Ergebnis ändert sich nicht.
# Es bleibt der Hinweis auf die nicht verwendeten Zahnweiten.
form = expected["root_form_diameter_mm"]
limited = pr.compute_pair_geometry(
    pair, root_form_diameter_mm=Pair(pinion=form["pinion"], wheel=form["wheel"])
)
assert {warning.code for warning in limited.warnings} == {"span_measurement_not_used"}
assert limited.sap_diameter_mm == result.sap_diameter_mm

# Ein größerer Fuß-Formkreis am Ritzel begrenzt die aktive Flanke und verkürzt die Eingriffsstrecke.
raised = pr.compute_pair_geometry(
    pair, root_form_diameter_mm=Pair(pinion=133.5, wheel=form["wheel"])
)
assert raised.sap_diameter_mm.pinion == 133.5
assert raised.active_tip_diameter_mm.wheel < result.active_tip_diameter_mm.wheel
pd.DataFrame(
    [
        {
            "d_Ff1 in mm": label,
            "d_Nf1 in mm": geometry.sap_diameter_mm.pinion,
            "d_Na2 in mm": geometry.active_tip_diameter_mm.wheel,
            "g_alpha in mm": geometry.length_of_path_of_contact_mm,
            "epsilon_alpha": geometry.transverse_contact_ratio,
            "Hinweise": ", ".join(dict.fromkeys(warning.code for warning in geometry.warnings)),
        }
        for label, geometry in (
            ("nicht gegeben", result),
            ("132,248 (Norm)", limited),
            ("133,5", raised),
        )
    ]
)

In [ ]:
def _plot_line_of_action(geometry):
    # Punkte auf der Eingriffsgeraden, gemessen ab T_1 (Krümmungsradius am Ritzel)
    d_b1 = geometry.gears.pinion.base_diameter_mm
    d_b2 = geometry.gears.wheel.base_diameter_mm
    alpha_wt = iv.transverse_profile_angle_at(geometry.working_pitch_diameter_mm.pinion, d_b1)
    length = pr.length_between_tangent_points(geometry.centre_distance_mm, alpha_wt)
    points = {
        "T_1": 0.0,
        "A": length - iv.radius_of_curvature(geometry.active_tip_diameter_mm.wheel, d_b2),
        "C": iv.radius_of_curvature(geometry.working_pitch_diameter_mm.pinion, d_b1),
        "E": iv.radius_of_curvature(geometry.active_tip_diameter_mm.pinion, d_b1),
        "T_2": length,
    }
    fig, ax = plt.subplots(figsize=(8, 1.8))
    ax.plot([0.0, length], [0.0, 0.0], color="gray", linewidth=1)
    ax.plot([points["A"], points["E"]], [0.0, 0.0], color="tab:blue", linewidth=4)
    for name, position in points.items():
        ax.plot([position], [0.0], "o", color="black")
        ax.annotate(
            f"${latex(name)}$",
            (position, 0.0),
            textcoords="offset points",
            xytext=(0, 9),
            ha="center",
        )
    ax.set_yticks([])
    ax.set_xlabel(
        f"Krümmungsradius am Ritzel ${latex('rho_y')}$ in mm (Abstand von ${latex('T_1')}$)"
    )
    ax.set_title(
        f"Eingriffsstrecke ${latex('g_alpha')}$ = {geometry.length_of_path_of_contact_mm:.3f} mm"
    )
    fig.tight_layout()
    return fig


_plot_line_of_action(result);

## 5 Kopfkreisdurchmesser als ausdrückliche Wahl

**Normbezug.** DIN ISO 21771:2014-08: §4.5.2, §4.5.3 Gl. (33), S. 34.

$$d_{\mathrm{a}} = d + 2\,(x\, m_{\mathrm{n}} + h_{\mathrm{aP}} + k\, m_{\mathrm{n}}) \quad (33)$$

$k$ ist der Kopfhöhenänderungsfaktor (vorzeichenbehaftet; negativ kürzt den Zahnkopf). Fehlt ein Kopfkreisdurchmesser, rechnet `gearcore` nicht mit einem angenommenen Wert, sondern meldet es. STplus setzt in diesem Fall $d_{\mathrm{a}} = d + 2\, m_{\mathrm{n}} (h_{\mathrm{aP}}^{*} + x)$ und verkleinert den Kopfkreis anschließend bei Bedarf (Programmanleitung S. 19; aufbewahrt in `data/stplus_program/defaults.yaml`).

In [ ]:
without_tips = pair.model_copy(
    update={
        "gears": Pair(
            pinion=gears["pinion"].model_copy(update={"tip_diameter_mm": None}),
            wheel=gears["wheel"].model_copy(update={"tip_diameter_mm": None}),
        )
    }
)
try:
    pr.compute_pair_geometry(without_tips)
except InputRangeError as error:
    print(error)

# ausdrückliche Wahl: Nennwert mit der Kopfhöhe des Bezugsprofils h_aP = 1 m_n (DIN 867, ISO 53)
nominal = pr.with_nominal_tip_diameters(without_tips, h_aP_mm=inputs["normal_module_mm"]["value"])
pd.DataFrame(
    [
        {
            "Rad": role,
            "d_a nach Gl. (33) in mm": getattr(nominal.gears, role).tip_diameter_mm,
            "d_a des Beispiels in mm": inputs["tip_diameter_mm"][role],
        }
        for role in ("pinion", "wheel")
    ]
)

## 6 Gleitverhältnisse

**Normbezug.** DIN ISO 21771:2014-08: §5.6.2 Gl. (112), (113), S. 54; §5.6.3 Gl. (114) bis (117), S. 54; Krümmungsradien §5.4.5.3 Gl. (83), (84), (87), S. 46.

$$K_{\mathrm{ga}} = \frac{2\, g_{\mathrm{a}}}{d_{\mathrm{w1}}}\left(1 + \frac{1}{u}\right) \quad (113) \qquad \zeta_{\mathrm{1}} = 1 - \frac{\rho_{\mathrm{y2}}}{u\, \rho_{\mathrm{y1}}} \quad (114) \qquad \zeta_{\mathrm{2}} = 1 - \frac{u\, \rho_{\mathrm{y1}}}{\rho_{\mathrm{y2}}} \quad (115)$$

Die Norm gibt die Gleitfaktoren für das treibende Ritzel an: $K_{\mathrm{ga}}$ im Endpunkt E (Gl. (113), mit $g_{\mathrm{a}}$) und $K_{\mathrm{gf}}$ im Anfangspunkt A (Gl. (112), mit $g_{\mathrm{f}}$). `gearcore` führt die Größen je Rad: Der Wert am Kopf des Rades ist das $K_{\mathrm{gf}}$ der Norm, weil $g_{\mathrm{a2}} = g_{\mathrm{f1}}$ gilt. Das spezifische Gleiten in den Endpunkten, $\zeta_{\mathrm{f1}}$ (Gl. (116)) und $\zeta_{\mathrm{f2}}$ (Gl. (117)), sind die Werte am Fuß des jeweiligen Rades. Am Grundkreis ist das spezifische Gleiten unbeschränkt. Liegt der Beginn der aktiven Flanke eines Rades auf seinem Grundkreis, genauer bei $d_{\mathrm{Nf}} \le d_{\mathrm{b}}\,(1 + 10^{-12})$, bricht `gearcore` deshalb mit einer Meldung ab; knapp darüber ist der Wert groß, aber endlich, und wird ausgegeben. Die Gleitfaktoren sind vorzeichenbehaftet wie $g_{\mathrm{a}}$ (negativ, wenn der aktive Kopfkreis innerhalb des Wälzkreises liegt).

In [ ]:
pd.DataFrame(
    [
        {
            "Rad": role,
            "g_a in mm": getattr(result.length_of_addendum_path_of_contact_mm, role),
            "Gleitfaktor am Zahnkopf": getattr(result.sliding_factor_at_tip, role),
            "spez. Gleiten am Zahnfuß": getattr(result.specific_sliding_at_end_points, role),
        }
        for role in ("pinion", "wheel")
    ]
)

In [ ]:
def _plot_specific_sliding(geometry):
    d_b1 = geometry.gears.pinion.base_diameter_mm
    d_b2 = geometry.gears.wheel.base_diameter_mm
    alpha_wt = iv.transverse_profile_angle_at(geometry.working_pitch_diameter_mm.pinion, d_b1)
    length = pr.length_between_tangent_points(geometry.centre_distance_mm, alpha_wt)
    start = length - iv.radius_of_curvature(geometry.active_tip_diameter_mm.wheel, d_b2)
    end = iv.radius_of_curvature(geometry.active_tip_diameter_mm.pinion, d_b1)
    steps = 200
    positions = [start + (end - start) * index / steps for index in range(steps + 1)]
    u = geometry.gear_ratio
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(
        positions,
        [pr.specific_sliding_of_pinion(p, length - p, u) for p in positions],
        label=f"Ritzel ${latex('zeta_1')}$",
    )
    ax.plot(
        positions,
        [pr.specific_sliding_of_wheel(p, length - p, u) for p in positions],
        label=f"Rad ${latex('zeta_2')}$",
    )
    ax.axhline(0.0, color="gray", linewidth=0.8)
    ax.set_xlabel(f"Krümmungsradius am Ritzel ${latex('rho_y')}$ in mm (A bis E)")
    ax.set_ylabel("spezifisches Gleiten")
    ax.legend()
    fig.tight_layout()
    return fig


_plot_specific_sliding(result);

## 7 Vergleich mit STplus

Alle Referenzfälle werden Wert für Wert verglichen (ADR-102, ADR-106). Was STplus selbst bestimmt hat, wird seiner genauesten Ausgabe entnommen: der Profilverschiebungsfaktor $x_{\mathrm{1}}$, wenn die Eingabe nur Zahnweiten nennt, selbst gewählte Kopfkreise und die Formkreise aus der Erzeugung. Die Rundung dieser Werte geht als Eingabetoleranz in jeden Vergleichswert ein.

Paarungsgrößen sind Differenzen und Quotienten größerer Zahlen. Die Genauigkeit eines Ergebnisses in einfacher Genauigkeit richtet sich dann nach der Größe der Operanden. Die Rechentoleranz eines Wertes ist deshalb eine Fehlerfortpflanzung erster Ordnung: Jede Zahl, die STplus hält, wird um zwei Schritte der einfachen Genauigkeit verschoben, und die Änderungen des Wertes werden summiert.

In [ ]:
rows = [row for case in stplus_case_dirs() for row in compare_pair_geometry(case.name)]
table = pd.DataFrame([row.model_dump(mode="json") for row in rows])
counts = table.groupby("verdict").size()
print(table["case"].nunique(), "Fälle,", len(table), "Vergleichswerte:", counts.to_dict())

documented = load_expected_differences()
different = table[table["verdict"] == ParityVerdict.DIFFERENT.value]
assert set(different["field"]) == {entry["field"] for entry in documented.values()}, (
    "jede Abweichung ist dokumentiert"
)
table.pivot_table(index="field", columns="verdict", values="case", aggfunc="count", fill_value=0)

**Was der Vergleich zeigen kann.** Die Toleranz eines Wertes, bezogen auf den Wert, ist die Nachweisgrenze des Vergleichs: Ein kleinerer relativer Fehler in `gearcore` bliebe unbemerkt. Die Schnittstellendatei (fünf Nachkommastellen, nur eigene Läufe) ist für die meisten Größen um ein bis zwei Größenordnungen schärfer als das Listing (drei Nachkommastellen); beim Betriebseingriffswinkel, den auch das Listing mit fünf Nachkommastellen druckt, und bei den Profilverschiebungsfaktoren ist der Unterschied kleiner. Die Tabelle zeigt je Größe den ungünstigsten Fall.

Ein Teil der Vergleichswerte wiederholt eine Eingabe des Vergleichs: ein Kopf-Formkreis ohne Kantenbruch, ein nicht begrenzter aktiver Kopfkreis, die gemeinsame Zahnbreite, eine Sprungüberdeckung von null. Solche Zeilen prüfen eine Entscheidung und keine Formel; sie werden gezählt.

In [ ]:
repeated = [row for case in stplus_case_dirs() for row in rows_repeating_an_input(case.name)]
print(len(repeated), "von", len(rows), "Vergleichswerten wiederholen eine Eingabe des Vergleichs")

limits = pd.Series(detection_limits(rows)).unstack()
limits.style.format("{:.1e}")

**Bewusste Abweichung von der Norm: gemeinsame Zahnhöhe.** DIN ISO 21771:2014-08 §5.2.6 Gl. (59), S. 41, und DIN 3960:1987-03 §4.2.6 Gl. (4.2.08), S. 31, definieren die gemeinsame Zahnhöhe $h_{\mathrm{w}}$ als Überlappung der **Kopfkreise**. STplus setzt in dieselbe Formel die Kopf-**Form**kreise ein. `gearcore` tut das ebenfalls (Entscheidung des Nutzers, ADR-112): Jenseits des Kopf-Formkreises, auf dem Kopfkantenbruch, trägt der Zahn nicht; die Höhe, über die sich Evolventenflanken überdecken können, endet an den Kopf-Formkreisen. Ohne Kantenbruch sind beide Werte gleich. Mit Kantenbruch nennt das Ergebnis den Wert nach dem Wortlaut der Norm als Hinweis.

In [ ]:
deviation = load_norm_deviations()["common_tooth_depth_with_tip_form_circles"]
print(
    "Norm    :",
    deviation["norm"]["source"],
    deviation["norm"]["eq"],
    "S.",
    deviation["norm"]["page"],
    "|",
    deviation["norm"]["formula"],
)
print(
    "DIN alt :",
    deviation["old_norm"]["source"],
    deviation["old_norm"]["eq"],
    "S.",
    deviation["old_norm"]["page"],
    "|",
    deviation["old_norm"]["formula"],
)
print("gearcore:", deviation["gearcore"]["formula"], "|", deviation["decision"])
print("STplus  :", deviation["stplus"]["formula"])

depth_rows = []
for case in stplus_case_dirs():
    data = pair_data(case.name)
    geometry = compute_pair_from_data(data, data.values)
    if "common_tooth_depth_with_tip_form_circles" not in [w.code for w in geometry.warnings]:
        continue
    printed = table[(table["case"] == case.name) & (table["field"] == "common_tooth_depth_mm")]
    depth_rows.append(
        {
            "Fall": case.name,
            "Gl. (59) mit Kopfkreisen in mm": pr.common_tooth_depth(
                geometry.tip_diameter_mm.pinion,
                geometry.tip_diameter_mm.wheel,
                geometry.centre_distance_mm,
            ),
            "gearcore (Kopf-Formkreise) in mm": geometry.common_tooth_depth_mm,
            "STplus": printed["stplus_token"].iloc[-1],
        }
    )
assert not different.empty or len(depth_rows) > 0
pd.DataFrame(depth_rows)

## 8 Rückverfolgbarkeit

Jede Rechenfunktion trägt ihren Normbezug als `@eq`-Angabe mit Quelle und Ausgabe, Gleichung, Abschnitt und Seite (ADR-103). Die Tabelle wird aus dem Paket gelesen.

In [ ]:
rows = []
for name, function in inspect.getmembers(pr, inspect.isfunction):
    if name.startswith("_") or function.__module__ != pr.__name__:
        continue
    if name in pr.EQ_EXEMPT:
        continue  # Orchestratoren rufen nur getragene Funktionen auf
    references = equations_of(function)
    assert references, name
    rows.append(
        {
            "Funktion": f"gearcore.pair.{name}",
            "Normbezug": "; ".join(str(reference) for reference in references),
        }
    )
pd.DataFrame(rows)